# 3 — Case study: a 56,394-cell cluster labelled *neutrophil*

This walks through the largest confirmed error the method found in a published
10.8M-cell atlas, and shows why correcting one cluster matters.

## What the label says, and what the markers say

The cluster is labelled **Neutrophilic granulocyte**. Its data-derived markers are
`FCN1`, `CD14`, `S100A12`, `MAFB` — and it has **no `FCGR3B` and no `ELANE`**, the two
genes a neutrophil cluster should not be missing. `FCN1`/`CD14`/`S100A12` is a
**classical monocyte**.

In [ ]:
from celltype_audit import Ontology, Resolver
o = Ontology.load()
r = Resolver(o)

asserted, how = r.resolve('Neutrophilic granulocyte', organ='Lung')
print(asserted, o.label(asserted), '   via:', how)

## Why the lineage sweep cannot see it

The sweep compares **lineage anchor sets**. Neutrophil and classical monocyte are both
haematopoietic, so the anchor sets overlap and no contradiction is raised. This is
structural: 9 of 14 known errors in the reference study sat inside a single lineage,
which is exactly why a second test exists.

In [ ]:
print('neutrophil        ', sorted(o.anchors('CL:0000775')))
print('classical monocyte', sorted(o.anchors('CL:0000860')))
print('conflict?         ', o.conflict('CL:0000775', 'CL:0000860'))

## What the marker queue sees

Scoring every CL term in lung against the cluster's markers puts **classical monocyte**
first. Note the asserted term still ranks 3rd with ratio 0.84 — neutrophils and
monocytes genuinely resemble each other — which is why the *winner's identity* is the
robust signal and the margin is not.

In [ ]:
from celltype_audit.reference import Reference
markers = ['FCN1', 'CD14', 'S100A12', 'MAFB', 'CSF3R', 'FCGR3B', 'ELANE']
ref = Reference.fetch(['UBERON:0002048'], markers)
sc = ref.score('UBERON:0002048', markers)
for c in sorted(sc, key=sc.get, reverse=True)[:5]:
    print('%-46s %.3f' % (o.label(c), sc[c]))

## Why it matters downstream

| | neutrophils | monocytes | ratio |
|---|---|---|---|
| atlas as published | 56,394 | 45,198 | **1.25 : 1** |
| after correcting one cluster | 0 | 101,592 | 0.00 : 1 |
| an independent atlas | 371 | 5,387 | **0.07 : 1** |

As published the atlas says neutrophils **outnumber** monocytes in human lung. An
independent atlas says the reverse, ~14:1 — an **18-fold** overstatement in a statistic
anyone might publish, removed by correcting a single cluster.

The biology agrees with the correction: neutrophils are fragile and RNA-poor and are
well known to be under-captured by droplet scRNA-seq, so a lung dataset in which they
are the commonest myeloid cell is the anomaly that needs explaining.

## Confirmed independently

Two sources that never saw this pipeline agree:

- **Tabula Sapiens** — scoring the cluster's markers across its lung cell types puts
  them on *mature NK T cell* / monocyte populations, not neutrophils
- **CellTypist** — run on the same cells, calls the cluster **NK cells** for the
  neighbouring tuft-cell case and *Alveolar fibroblasts* for the smooth-muscle case

Eight such errors were confirmed cross-atlas in total; none was refuted.